In [2]:
import os
os.chdir(r'D:\8th Semester\Machine Learning Lab\data parisr')

In [3]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, explained_variance_score, r2_score
from timeseires.utils.to_split import to_split
from timeseires.utils.multivariate_multi_step import multivariate_multi_step
from timeseires.utils.multivariate_single_step import multivariate_single_step
from timeseires.utils.univariate_multi_step import univariate_multi_step
from timeseires.utils.univariate_single_step import univariate_single_step
from timeseires.utils.CosineAnnealingLRS import CosineAnnealingLRS
from timeseires.callbacks.EpochCheckpoint import EpochCheckpoint
from tensorflow.keras.callbacks import ModelCheckpoint
from timeseires.callbacks.TrainingMonitor import TrainingMonitor
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.optimizers import SGD
from tensorflow.keras.models import load_model
from tensorflow.keras.layers import LSTM, Bidirectional, Add
from tensorflow.keras.layers import BatchNormalization
from tensorflow.keras.layers import Conv1D,TimeDistributed
from tensorflow.keras.layers import Dense, Dropout, Activation, Flatten,MaxPooling1D,Concatenate,AveragePooling1D, GlobalMaxPooling1D, Input
from tensorflow.keras.models import Sequential,Model
import pandas as pd
import time, pickle
import numpy as np
import tensorflow.keras.backend as K
import tensorflow
from tensorflow.keras.layers import Input, Reshape, Lambda
from tensorflow.keras.layers import Layer, Flatten, LeakyReLU, concatenate, Dense
from tensorflow.keras.regularizers import l2
import glob
import h5py
import matplotlib.pyplot as plt
from keras.callbacks import Callback

In [4]:
#lookback = 24
model = None
start_epoch = 0
time_steps=24
num_features=21

In [5]:
def CNN():
    input_data = Input(shape=(time_steps, num_features))
    x1 = Conv1D(16, 2, activation="relu")(input_data)
    x2 = Conv1D(16, 2, activation="relu")(x1)
    flatten = Flatten()(x2)
    output_data = Dense(1)(flatten)
    model = Model(input_data, output_data)
    return model

In [6]:
model1 = CNN()
model1.summary()


Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_1 (InputLayer)        [(None, 24, 21)]          0         
                                                                 
 conv1d (Conv1D)             (None, 23, 16)            688       
                                                                 
 conv1d_1 (Conv1D)           (None, 22, 16)            528       
                                                                 
 flatten (Flatten)           (None, 352)               0         
                                                                 
 dense (Dense)               (None, 1)                 353       
                                                                 
Total params: 1569 (6.13 KB)
Trainable params: 1569 (6.13 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [7]:
tensorflow.keras.utils.plot_model(model1 )

You must install pydot (`pip install pydot`) and install graphviz (see instructions at https://graphviz.gitlab.io/download/) for plot_model to work.


In [8]:
checkpoints = r'D:\8th Semester\Machine Learning Lab\data parisr-cp-{epoch:04d}-loss{val_loss:.2f}.h5'
OUTPUT_PATH = r'D:\8th Semester\Machine Learning Lab\New folder'
FIG_PATH = os.path.sep.join([OUTPUT_PATH,"\history.png"])
JSON_PATH = os.path.sep.join([OUTPUT_PATH,"\history.json"])

In [9]:
# construct the callback to save only the *best* model to disk
# based on the validation loss
EpochCheckpoint1 = ModelCheckpoint(checkpoints,
                             monitor="val_loss",
                             save_best_only=True, 
                             verbose=1)
TrainingMonitor1=TrainingMonitor(FIG_PATH, jsonPath=JSON_PATH, startAt=start_epoch)

# construct the set of callbacks
callbacks = [EpochCheckpoint1,TrainingMonitor1]

In [10]:
# if there is no specific model checkpoint supplied, then initialize
# the network and compile the model
if model is None:
    print("[INFO] compiling model...")
    model =CNN()
    opt = Adam(1e-3)
    model.compile(loss= 'mae', optimizer=opt, metrics=["mae", "mape"])
# otherwise, load the checkpoint from disk
else:
    print("[INFO] loading {}...".format(model))
    model = load_model(model)

    # update the learning rate
    print("[INFO] old learning rate: {}".format(K.get_value(model.optimizer.lr)))
    K.set_value(model.optimizer.lr, 1e-4)
    print("[INFO] new learning rate: {}".format(K.get_value(model.optimizer.lr)))

[INFO] compiling model...


In [12]:
import os
path_dataset =r'D:\8th Semester\Machine Learning Lab\data parisr'
path_tr = os.path.join(path_dataset, 'train.csv')
df_tr = pd.read_csv(path_tr)
train_set = df_tr.iloc[:].values
path_v = os.path.join(path_dataset, 'validation.csv')
df_v = pd.read_csv(path_v)
validation_set = df_v.iloc[:].values 
path_te = os.path.join(path_dataset, 'test.csv')
df_te = pd.read_csv(path_te)
test_set = df_te.iloc[:].values 

path_scaler = os.path.join(path_dataset, 'AEP_scaler.pkl')
scaler         = pickle.load(open(path_scaler, 'rb'))

train_set.shape, validation_set.shape, test_set.shape

c:\Users\Shafiq\.conda\envs\DSP\Lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator MinMaxScaler from version 1.0.2 when using version 1.7.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


((860, 21), (90, 21), (30, 21))

In [39]:
time_steps=24
num_features=21

In [15]:
start = time.time()
train_X , train_y = univariate_multi_step(train_set, time_steps, target_col=0,target_len=1)
validation_X, validation_y = univariate_multi_step(validation_set, time_steps, target_col=0,target_len=1)
test_X, test_y = univariate_multi_step(test_set, time_steps, target_col=0,target_len=1)
print('Time Consumed', time.time()-start, "sec")

Time Consumed 0.03920578956604004 sec


In [38]:
epochs = 60
verbose = 1 #0
batch_size = 32
History = model.fit(train_X,
                        train_y,
                        batch_size=batch_size,   
                        epochs = epochs, 
                        validation_data = (validation_X,validation_y),
                        callbacks=callbacks,verbose = verbose)

Epoch 1/60
22/27 [=======================>......] - ETA: 0s - loss: 0.0256 - mae: 0.0256 - mape: 9.6981 
Epoch 1: val_loss did not improve from 0.02801
27/27 [==============================] - 0s 18ms/step - loss: 0.0252 - mae: 0.0252 - mape: 10.0620 - val_loss: 0.0286 - val_mae: 0.0286 - val_mape: 9.2998
Epoch 2/60
16/27 [================>.............] - ETA: 0s - loss: 0.0251 - mae: 0.0251 - mape: 10.8335
Epoch 2: val_loss did not improve from 0.02801
27/27 [==============================] - 1s 27ms/step - loss: 0.0247 - mae: 0.0247 - mape: 9.9979 - val_loss: 0.0321 - val_mae: 0.0321 - val_mape: 10.8457
Epoch 3/60
23/27 [========================>.....] - ETA: 0s - loss: 0.0244 - mae: 0.0244 - mape: 9.7026
Epoch 3: val_loss did not improve from 0.02801
27/27 [==============================] - 1s 25ms/step - loss: 0.0244 - mae: 0.0244 - mape: 9.9302 - val_loss: 0.0349 - val_mae: 0.0349 - val_mape: 11.7804
Epoch 4/60
17/27 [=================>............] - ETA: 0s - loss: 0.0258 - mae

c:\Users\Shafiq\.conda\envs\DSP\Lib\site-packages\keras\src\engine\training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


27/27 [==============================] - 1s 21ms/step - loss: 0.0233 - mae: 0.0233 - mape: 10.2641 - val_loss: 0.0273 - val_mae: 0.0273 - val_mape: 8.3437
Epoch 11/60
19/27 [====================>.........] - ETA: 0s - loss: 0.0239 - mae: 0.0239 - mape: 10.9497
Epoch 11: val_loss did not improve from 0.02732
27/27 [==============================] - 1s 20ms/step - loss: 0.0238 - mae: 0.0238 - mape: 10.2109 - val_loss: 0.0303 - val_mae: 0.0303 - val_mape: 9.2028
Epoch 12/60
25/27 [==========================>...] - ETA: 0s - loss: 0.0240 - mae: 0.0240 - mape: 10.6525
Epoch 12: val_loss did not improve from 0.02732
27/27 [==============================] - 1s 26ms/step - loss: 0.0238 - mae: 0.0238 - mape: 10.5020 - val_loss: 0.0282 - val_mae: 0.0282 - val_mape: 8.4602
Epoch 13/60
22/27 [=======================>......] - ETA: 0s - loss: 0.0236 - mae: 0.0236 - mape: 10.6117
Epoch 13: val_loss did not improve from 0.02732
27/27 [==============================] - 0s 15ms/step - loss: 0.0234 - ma

c:\Users\Shafiq\.conda\envs\DSP\Lib\site-packages\keras\src\engine\training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


27/27 [==============================] - 0s 15ms/step - loss: 0.0221 - mae: 0.0221 - mape: 9.5349 - val_loss: 0.0262 - val_mae: 0.0262 - val_mape: 8.0623
Epoch 37/60
25/27 [==========================>...] - ETA: 0s - loss: 0.0218 - mae: 0.0218 - mape: 9.4041
Epoch 37: val_loss did not improve from 0.02623
27/27 [==============================] - 0s 16ms/step - loss: 0.0218 - mae: 0.0218 - mape: 9.2833 - val_loss: 0.0275 - val_mae: 0.0275 - val_mape: 8.1942
Epoch 38/60
24/27 [=========================>....] - ETA: 0s - loss: 0.0224 - mae: 0.0224 - mape: 9.5332
Epoch 38: val_loss did not improve from 0.02623
27/27 [==============================] - 1s 21ms/step - loss: 0.0224 - mae: 0.0224 - mape: 9.4274 - val_loss: 0.0265 - val_mae: 0.0265 - val_mape: 8.2216
Epoch 39/60
25/27 [==========================>...] - ETA: 0s - loss: 0.0225 - mae: 0.0225 - mape: 9.3440 
Epoch 39: val_loss did not improve from 0.02623
27/27 [==============================] - 1s 25ms/step - loss: 0.0228 - mae: 0.

...

27/27 [==============================] - 1s 22ms/step - loss: 0.0221 - mae: 0.0221 - mape: 9.1999 - val_loss: 0.0250 - val_mae: 0.0250 - val_mape: 7.3185
Epoch 50/60
18/27 [===================>..........] - ETA: 0s - loss: 0.0210 - mae: 0.0210 - mape: 8.5618
Epoch 50: val_loss did not improve from 0.02502
27/27 [==============================] - 1s 21ms/step - loss: 0.0216 - mae: 0.0216 - mape: 8.9020 - val_loss: 0.0277 - val_mae: 0.0277 - val_mape: 8.5497
Epoch 51/60
14/27 [==============>...............] - ETA: 0s - loss: 0.0230 - mae: 0.0230 - mape: 8.8016
Epoch 51: val_loss did not improve from 0.02502
27/27 [==============================] - 0s 17ms/step - loss: 0.0220 - mae: 0.0220 - mape: 9.0530 - val_loss: 0.0292 - val_mae: 0.0292 - val_mape: 9.3621
Epoch 52/60
17/27 [=================>............] - ETA: 0s - loss: 0.0231 - mae: 0.0231 - mape: 10.0997
Epoch 52: val_loss did not improve from 0.02502
27/27 [==============================] - 0s 14ms/step - loss: 0.0230 - mae: 0.

c:\Users\Shafiq\.conda\envs\DSP\Lib\site-packages\keras\src\engine\training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


27/27 [==============================] - 1s 27ms/step - loss: 0.0228 - mae: 0.0228 - mape: 9.6482 - val_loss: 0.0249 - val_mae: 0.0249 - val_mape: 7.1746
Epoch 56/60
27/27 [==============================] - ETA: 0s - loss: 0.0214 - mae: 0.0214 - mape: 9.1809
Epoch 56: val_loss did not improve from 0.02494
27/27 [==============================] - 1s 24ms/step - loss: 0.0214 - mae: 0.0214 - mape: 9.1809 - val_loss: 0.0275 - val_mae: 0.0275 - val_mape: 8.7023
Epoch 57/60
19/27 [====================>.........] - ETA: 0s - loss: 0.0204 - mae: 0.0204 - mape: 8.5055 
Epoch 57: val_loss improved from 0.02494 to 0.02476, saving model to D:\8th Semester\Machine Learning Lab\New folder-cp-0057-loss0.02.h5


c:\Users\Shafiq\.conda\envs\DSP\Lib\site-packages\keras\src\engine\training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


27/27 [==============================] - 0s 15ms/step - loss: 0.0212 - mae: 0.0212 - mape: 9.0529 - val_loss: 0.0248 - val_mae: 0.0248 - val_mape: 6.9998
Epoch 58/60
17/27 [=================>............] - ETA: 0s - loss: 0.0229 - mae: 0.0229 - mape: 9.0659
Epoch 58: val_loss did not improve from 0.02476
27/27 [==============================] - 0s 18ms/step - loss: 0.0222 - mae: 0.0222 - mape: 9.2176 - val_loss: 0.0269 - val_mae: 0.0269 - val_mape: 8.6079
Epoch 59/60
16/27 [================>.............] - ETA: 0s - loss: 0.0220 - mae: 0.0220 - mape: 9.3635 
Epoch 59: val_loss did not improve from 0.02476
27/27 [==============================] - 1s 21ms/step - loss: 0.0216 - mae: 0.0216 - mape: 8.9934 - val_loss: 0.0275 - val_mae: 0.0275 - val_mape: 8.3548
Epoch 60/60
23/27 [========================>.....] - ETA: 0s - loss: 0.0224 - mae: 0.0224 - mape: 9.5265 
Epoch 60: val_loss did not improve from 0.02476
27/27 [==============================] - 1s 22ms/step - loss: 0.0222 - mae: 0

In [22]:

model = load_model(r'D:\8th Semester\Machine Learning Lab\data parisr\E1-cp-0001-loss0.12.h5')

y_pred_scaled   = model.predict(test_X)
y_pred          = scaler.inverse_transform(y_pred_scaled)
y_test_unscaled = scaler.inverse_transform(test_y)
# Mean Absolute Error (MAE)
MAE = np.mean(abs(y_pred - y_test_unscaled)) 
print('Mean Absolute Error (MAE): ' + str(np.round(MAE, 2)))

# Median Absolute Error (MedAE)
MEDAE = np.median(abs(y_pred - y_test_unscaled))
print('Median Absolute Error (MedAE): ' + str(np.round(MEDAE, 2)))

# Mean Squared Error (MSE)
MSE = np.square(np.subtract(y_pred, y_test_unscaled)).mean()
print('Mean Squared Error (MSE): ' + str(np.round(MSE, 2)))

# Root Mean Squarred Error (RMSE) 
RMSE = np.sqrt(np.mean(np.square(y_pred - y_test_unscaled)))
print('Root Mean Squared Error (RMSE): ' + str(np.round(RMSE, 2)))

# Mean Absolute Percentage Error (MAPE)
MAPE = np.mean((np.abs(np.subtract(y_test_unscaled, y_pred)/ y_test_unscaled))) * 100
print('Mean Absolute Percentage Error (MAPE): ' + str(np.round(MAPE, 2)) + ' %')

# Median Absolute Percentage Error (MDAPE)
MDAPE = np.median((np.abs(np.subtract(y_test_unscaled, y_pred)/ y_test_unscaled))) * 100
print('Median Absolute Percentage Error (MDAPE): ' + str(np.round(MDAPE, 2)) + ' %')

print('\n\ny_test_unscaled.shape= ',y_test_unscaled.shape)
print('y_pred.shape= ',y_pred.shape)

1/1 [==============================] - 0s 227ms/step
Mean Absolute Error (MAE): 12942.4
Median Absolute Error (MedAE): 12955.77
Mean Squared Error (MSE): 168205231.45
Root Mean Squared Error (RMSE): 12969.4
Mean Absolute Percentage Error (MAPE): 82.69 %
Median Absolute Percentage Error (MDAPE): 81.79 %


y_test_unscaled.shape=  (5, 1)
y_pred.shape=  (5, 1)


In [28]:
checkpoints = r'D:\8th Semester\Machine Learning Lab\New folder-cp-{epoch:04d}-loss{val_loss:.2f}.h5'
model=r'D:\8th Semester\Machine Learning Lab\data parisr\E1-cp-0060-loss0.03.h5'
start_epoch= 58

In [29]:
# construct the callback to save only the *best* model to disk
# based on the validation loss
EpochCheckpoint1 = ModelCheckpoint(checkpoints,
                             monitor="val_loss",
                             save_best_only=True, 
                             verbose=1)
TrainingMonitor1=TrainingMonitor(FIG_PATH, jsonPath=JSON_PATH, startAt=start_epoch)

# construct the set of callbacks
callbacks = [EpochCheckpoint1,TrainingMonitor1]
# if there is no specific model checkpoint supplied, then initialize
# the network and compile the model
if model is None:
    print("[INFO] compiling model...")
    model = PC.build(time_steps=24, num_features=21, reg=0.0005)
    opt = Adam(1e-3)
    model.compile(loss= 'mae', optimizer=opt, metrics=["mae", "mape"])
# otherwise, load the checkpoint from disk
else:
    print("[INFO] loading {}...".format(model))
    model = load_model(model)

    # update the learning rate
    print("[INFO] old learning rate: {}".format(K.get_value(model.optimizer.lr)))
    K.set_value(model.optimizer.lr, 1e-4)
    print("[INFO] new learning rate: {}".format(K.get_value(model.optimizer.lr)))

[INFO] loading D:\8th Semester\Machine Learning Lab\data parisr\E1-cp-0060-loss0.03.h5...
[INFO] old learning rate: 0.0010000000474974513
[INFO] new learning rate: 9.999999747378752e-05


In [35]:
epochs = 10
verbose = 1 #0
batch_size = 32
History = model.fit(train_X,
                        train_y,
                        batch_size=batch_size,   
                        epochs = epochs, 
                        validation_data = (validation_X,validation_y),
                        callbacks=callbacks,
                        verbose = verbose)

Epoch 1/10
22/27 [=======================>......] - ETA: 0s - loss: 0.0381 - mae: 0.0381 - mape: 13.7853
Epoch 1: val_loss did not improve from 0.02801
27/27 [==============================] - 1s 26ms/step - loss: 0.0372 - mae: 0.0372 - mape: 13.5535 - val_loss: 0.0366 - val_mae: 0.0366 - val_mape: 11.9973
Epoch 2/10
23/27 [========================>.....] - ETA: 0s - loss: 0.0313 - mae: 0.0313 - mape: 12.4837
Epoch 2: val_loss did not improve from 0.02801
27/27 [==============================] - 1s 24ms/step - loss: 0.0311 - mae: 0.0311 - mape: 12.5689 - val_loss: 0.0443 - val_mae: 0.0443 - val_mape: 14.9989
Epoch 3/10
20/27 [=====================>........] - ETA: 0s - loss: 0.0295 - mae: 0.0295 - mape: 11.2158
Epoch 3: val_loss did not improve from 0.02801
27/27 [==============================] - 1s 22ms/step - loss: 0.0293 - mae: 0.0293 - mape: 11.3916 - val_loss: 0.0382 - val_mae: 0.0382 - val_mape: 12.7210
Epoch 4/10
21/27 [======================>.......] - ETA: 0s - loss: 0.0289 -

In [31]:

model = load_model(r'D:\8th Semester\Machine Learning Lab\data parisr\E1-cp-0001-loss0.12.h5')

y_pred_scaled   = model.predict(test_X)
y_pred          = scaler.inverse_transform(y_pred_scaled)
y_test_unscaled = scaler.inverse_transform(test_y)
# Mean Absolute Error (MAE)
MAE = np.mean(abs(y_pred - y_test_unscaled)) 
print('Mean Absolute Error (MAE): ' + str(np.round(MAE, 2)))

# Median Absolute Error (MedAE)
MEDAE = np.median(abs(y_pred - y_test_unscaled))
print('Median Absolute Error (MedAE): ' + str(np.round(MEDAE, 2)))

# Mean Squared Error (MSE)
MSE = np.square(np.subtract(y_pred, y_test_unscaled)).mean()
print('Mean Squared Error (MSE): ' + str(np.round(MSE, 2)))

# Root Mean Squarred Error (RMSE) 
RMSE = np.sqrt(np.mean(np.square(y_pred - y_test_unscaled)))
print('Root Mean Squared Error (RMSE): ' + str(np.round(RMSE, 2)))

# Mean Absolute Percentage Error (MAPE)
MAPE = np.mean((np.abs(np.subtract(y_test_unscaled, y_pred)/ y_test_unscaled))) * 100
print('Mean Absolute Percentage Error (MAPE): ' + str(np.round(MAPE, 2)) + ' %')

# Median Absolute Percentage Error (MDAPE)
MDAPE = np.median((np.abs(np.subtract(y_test_unscaled, y_pred)/ y_test_unscaled))) * 100
print('Median Absolute Percentage Error (MDAPE): ' + str(np.round(MDAPE, 2)) + ' %')

print('\n\ny_test_unscaled.shape= ',y_test_unscaled.shape)
print('y_pred.shape= ',y_pred.shape)

1/1 [==============================] - 0s 159ms/step
Mean Absolute Error (MAE): 12942.4
Median Absolute Error (MedAE): 12955.77
Mean Squared Error (MSE): 168205231.45
Root Mean Squared Error (RMSE): 12969.4
Mean Absolute Percentage Error (MAPE): 82.69 %
Median Absolute Percentage Error (MDAPE): 81.79 %


y_test_unscaled.shape=  (5, 1)
y_pred.shape=  (5, 1)


# lab report 

## Lab 1